In [24]:
from embedders import SentenceTransformerEmbedder

print("Inicializando módulos...")
embedder = SentenceTransformerEmbedder()

c:\Users\omarinf\Documents\TFM\alert_correlation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Inicializando módulos...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9921.53it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [1]:
import os
from mitreattack.stix20 import MitreAttackData


# import requests
# url = "https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json"
# url2 = "https://raw.githubusercontent.com/mitre-attackenterprise-attack.json"
# response = requests.get(url)
# if response.status_code == 200:
#     with open("enterprise-attack.json", "w") as f:
#         f.write(response.text)
#     print("Archivo MITRE ATT&CK descargado exitosamente.")
# else:
#     print(f"Error al descargar el archivo: {response.status_code}")

stix_filepath = os.environ.get("STIX_BUNDLE", "enterprise-attack.json")
print(f"Cargando datos de MITRE ATT&CK desde: {stix_filepath}")
mitre_data = MitreAttackData(stix_filepath=stix_filepath)
# mitre_data = MitreAttackData('enterprise-attack')
techniques = mitre_data.get_techniques()

Cargando datos de MITRE ATT&CK desde: enterprise-attack.json


In [20]:
len(techniques)

835

In [23]:
cleaned_techniques = []
for t in techniques:
    if t.revoked or t.x_mitre_deprecated:
        continue
    cleaned_techniques.append(t)

len(cleaned_techniques)

691

In [3]:
t0 = techniques[0]
print(f"Técnica: {t0.name}")
print(f"Descripción: {t0.description}")
print(f'ID: {t0.id}')
print(f'technique_id: {mitre_data.get_attack_id(t0.id)}')
print(f'Tactics: {[phase.phase_name for phase in t0.kill_chain_phases]}')
print(f'Platforms: {t0.x_mitre_platforms}')

Técnica: Extra Window Memory Injection
Descripción: Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/output of data).(Citation: Microsoft Window Classes) Registration of new windows classes can include a request for up to 40 bytes of EWM to be appended to the allocated memory of each instance of that class. This EWM is intended to store data specific to that window and has specific application programming interface (API) functions to set and get its value. (Citation: Microsoft GetWindowLong function) (Citation: Microsoft SetWindowLong function)

Although small, the 

In [9]:
for k,v in dict(t0).items():
    print(f"{k}: {v}")

type: attack-pattern
id: attack-pattern--0042a9f5-f053-4769-b3ef-9ad018dfa298
created_by_ref: identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5
created: 2020-01-14 17:18:32.126000+00:00
modified: 2025-10-24 17:48:19.059000+00:00
name: Extra Window Memory Injection
description: Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/output of data).(Citation: Microsoft Window Classes) Registration of new windows classes can include a request for up to 40 bytes of EWM to be appended to the allocated memory of each instance of that class. This EWM is intended to store data speci

In [16]:
t0.external_references[2].serialize()

'{"source_name": "Microsoft GetWindowLong function", "description": "Microsoft. (n.d.). GetWindowLong function. Retrieved December 16, 2017.", "url": "https://msdn.microsoft.com/library/windows/desktop/ms633584.aspx"}'

In [18]:
t = techniques[0]
doc = {
    'id': t.id,
    'technique_id': mitre_data.get_attack_id(t.id),
    'name': t.name,
    'description': t.description,
    'tactics': [phase.phase_name for phase in t.kill_chain_phases],
    'platforms': t.x_mitre_platforms,
    'is_subtechnique': t.x_mitre_is_subtechnique,
    'detection': t.x_mitre_detection,
    'external_references': [dict(ref) for ref in t.external_references if ref],
    # 'vector': embedder.embed_query(t.description)
}

print("\nDocumento estructurado para indexación:")
for k, v in doc.items():
    print(f"{k}: {v}")


Documento estructurado para indexación:
id: attack-pattern--0042a9f5-f053-4769-b3ef-9ad018dfa298
technique_id: T1055.011
name: Extra Window Memory Injection
description: Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/output of data).(Citation: Microsoft Window Classes) Registration of new windows classes can include a request for up to 40 bytes of EWM to be appended to the allocated memory of each instance of that class. This EWM is intended to store data specific to that window and has specific application programming interface (API) functions to set and get its

In [5]:
# Ver todas las columnas disponibles
print("\nColumnas disponibles en la técnica:")
for key in t0.__dict__.keys():
    print(f"- {key}")


Columnas disponibles en la técnica:
- _STIXBase__now
- _STIXBase__INTEROPERABILITY_types
- _defaulted_optional_properties
- _inner
- _STIXBase__has_custom


In [21]:
from store import ElasticSearchVectorStore
vector_store = ElasticSearchVectorStore(index_name='mitre_attack')

In [22]:
vector_store.client.indices.exists(index=vector_store.index_name)

True

In [25]:
query = 'escalada de privilegios en Windows'

vector_query = embedder.embed_query(query)

vector_store.search(vector_query)

[{'id': 'attack-pattern--4ff5d6a8-c062-4c68-a778-36fc5edd564f',
  'technique_id': 'T1218.002',
  'name': 'Control Panel',
  'description': 'Adversaries may abuse control.exe to proxy execution of malicious payloads. The Windows Control Panel process binary (control.exe) handles execution of Control Panel items, which are utilities that allow users to view and adjust computer settings.\n\nControl Panel items are registered executable (.exe) or Control Panel (.cpl) files, the latter are actually renamed dynamic-link library (.dll) files that export a <code>CPlApplet</code> function.(Citation: Microsoft Implementing CPL)(Citation: TrendMicro CPL Malware Jan 2014) For ease of use, Control Panel items typically include graphical menus available to users after being registered and loaded into the Control Panel.(Citation: Microsoft Implementing CPL) Control Panel items can be executed directly from the command line, programmatically via an application programming interface (API) call, or by s